# Exercises XP – Week 6, Day 1: Training Deep Neural Networks

## 🌟 Exercise 1: Deep Learning vs. Traditional Machine Learning

| Aspect | Traditional Machine Learning | Deep Learning |
|---|---|---|
| **Feature Engineering** | Mostly **manual**: experts decide which features to create (ratios, counts, edges in an image). | Mostly **automatic**: the network learns its own features from raw data, layer by layer. |
| **Data Processing** | Works best with **structured, tabular** data (rows and columns). | Handles **unstructured** data well: images, audio, text, video. |
| **Scalability** | Performance usually **plateaus** after a certain amount of data. | Performance keeps **improving with more data** and bigger models. |
| **Pattern Discovery** | Finds simpler, often near-linear patterns; relies on the features it's given. | Discovers **complex, non-linear, hierarchical** patterns (e.g. pixels → edges → shapes → objects). |
| **Computational Requirements** | Low to moderate: trains on a normal CPU, often in seconds or minutes. | High: usually needs **GPUs/TPUs**, lots of memory, and hours or days of training for large models. |

**Where each is the better choice:**
- **Traditional ML is better suited for credit-risk scoring** on a bank's tabular data (income, debts, credit history). The data is structured and limited in size, and the decisions must be **explainable** to customers and regulators. A logistic regression or gradient-boosted tree is accurate, fast and interpretable.
- **Deep learning is superior for medical image analysis**, such as detecting tumours in X-rays or MRI scans. The input is raw pixels, the patterns are subtle and visual, and convolutional neural networks can learn them directly from many labelled images.

**Why deep learning has an advantage with unstructured data:** unstructured data like images, sound or text has no ready-made columns, and the useful information is hidden in complex relationships between thousands of raw values (pixels, audio samples, words). Traditional models need people to hand-craft features from this data, which is slow and usually misses important patterns. Deep networks learn **hierarchical representations** automatically: early layers detect simple patterns (edges, sounds, word pieces), and deeper layers combine them into complex concepts (faces, words, meaning). Because these features are learned from the data itself, deep learning keeps improving as more examples are added, which is why it dominates computer vision, speech recognition and language tasks.

## 🌟 Exercise 2: Artificial Neural Networks (ANNs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

layers = {"Input layer\n(3 neurons)": 3, "Hidden layer\n(4 neurons)": 4, "Output layer\n(2 neurons)": 2}
x_positions = [0, 3, 6]
colors = ["#9ecae1", "#a1d99b", "#fdae6b"]

fig, ax = plt.subplots(figsize=(11, 7))
coords = []
for x, (name, n), color in zip(x_positions, layers.items(), colors):
    ys = np.linspace(-(n - 1) / 2, (n - 1) / 2, n) * 1.6
    coords.append([(x, y) for y in ys])
    ax.text(x, max(ys) + 1.3, name, ha="center", fontsize=11, fontweight="bold")

# Connections = weights
for layer_a, layer_b in zip(coords[:-1], coords[1:]):
    for (x1, y1) in layer_a:
        for (x2, y2) in layer_b:
            ax.plot([x1, x2], [y1, y2], color="grey", linewidth=0.8, zorder=1)

# Neurons
for layer, color in zip(coords, colors):
    for (x, y) in layer:
        ax.add_patch(plt.Circle((x, y), 0.35, color=color, ec="black", zorder=2))

# Labels for the components
ax.annotate("weight (w)\none per connection", xy=(1.5, -0.9), xytext=(-2.9, -3.3),
            arrowprops=dict(arrowstyle="->"), fontsize=10)
ax.annotate("neuron", xy=(0, 1.6), xytext=(-2.2, 2.8), arrowprops=dict(arrowstyle="->"), fontsize=10)
ax.annotate("each hidden / output neuron:\nz = Σ(w·x) + bias (b)\noutput = activation(z), e.g. ReLU",
            xy=(3, -2.4), xytext=(1.0, -5.2), arrowprops=dict(arrowstyle="->"), fontsize=10)
ax.annotate("output activation\n(e.g. softmax / sigmoid)", xy=(6, -0.8), xytext=(6.6, -3.6),
            arrowprops=dict(arrowstyle="->"), fontsize=10)
for x, label in zip(x_positions, ["x₁, x₂, x₃", "h₁ … h₄", "ŷ₁, ŷ₂"]):
    ax.text(x, -4.0 if x != 6 else -2.2, label, ha="center", fontsize=10, style="italic")

ax.set_xlim(-3, 9)
ax.set_ylim(-6, 5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("A simple ANN: 3 inputs → 4 hidden neurons → 2 outputs", fontsize=13)
plt.show()

**How information flows through the network:** the **input layer** receives the raw values (x₁, x₂, x₃) and passes them forward. Each neuron in the **hidden layer** multiplies every input by its own **weight**, adds them up together with a **bias**, and passes the result through an **activation function** (such as ReLU), which adds non-linearity. The hidden layer's outputs then become the inputs of the **output layer**, where the same process happens again, usually with a softmax or sigmoid activation that turns the results into probabilities for the 2 outputs. This one-directional pass from input to output is called **forward propagation**. During training, the error of the output is sent backwards (**backpropagation**) to adjust the weights and biases.

## 🌟 Exercise 3: Creating the Dataset and Visualizing the Data

In [ ]:
from sklearn.metrics import mean_squared_error

np.random.seed(0)
x = np.arange(-1, 1, 0.1)
y = -x**2 + np.random.normal(0, 0.05, len(x))
print("Number of points:", len(x))

plt.figure(figsize=(7, 5))
plt.scatter(x, y, color="steelblue")
plt.title("Dataset: y = −x² + noise")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

# Training set = first 12 points, test set = last 8 points
x_train, y_train = x[:12], y[:12]
x_test, y_test = x[12:], y[12:]
print("Train:", len(x_train), "points | Test:", len(x_test), "points")

## 🌟 Exercise 4: Fitting Polynomial Models of Different Degrees

In [ ]:
def polynomial_fit(degree):
    """Fit a polynomial of the given degree to the training data and return it as a function."""
    coefficients = np.polyfit(x_train, y_train, degree)
    return np.poly1d(coefficients)

def plot_polyfit(degree):
    p = polynomial_fit(degree)
    x_curve = np.linspace(-1, 1, 300)
    plt.figure(figsize=(7, 5))
    plt.scatter(x_train, y_train, color="steelblue", label="Training set")
    plt.scatter(x_test, y_test, color="darkorange", label="Test set")
    plt.plot(x_curve, p(x_curve), color="green", label=f"Polynomial degree {degree}")
    plt.ylim(-1.5, 0.5)
    plt.title(f"Polynomial fit – degree {degree}")
    plt.legend()
    plt.show()

for degree in [1, 7, 11]:
    plot_polyfit(degree)

**Observations:**
- **Degree 1** (a straight line) is too simple. It can't follow the curve, so it fits both training and test points poorly. This is **underfitting**.
- **Degree 7** follows the training points very closely, including the noise, and starts to bend strangely outside them.
- **Degree 11** passes almost exactly through every training point, but between and beyond them it swings wildly, so its predictions on the test set are very wrong. This is **overfitting**: the model memorised the noise instead of learning the real pattern.

As the degree increases, the training error keeps going down, but the test error first goes down and then shoots up.

## 🌟 Exercise 5: Cross-Validation to Find the Optimal Degree

In [ ]:
results = []
for degree in range(1, 12):
    p = polynomial_fit(degree)
    rmse_train = np.sqrt(mean_squared_error(y_train, p(x_train)))
    rmse_test = np.sqrt(mean_squared_error(y_test, p(x_test)))
    results.append((degree, rmse_train, rmse_test))

for degree, tr, te in results:
    print(f"Degree {degree:2d}: train RMSE = {tr:.4f} | test RMSE = {te:.4f}")

degrees, rmse_train_list, rmse_test_list = zip(*results)
plt.figure(figsize=(8, 5))
plt.plot(degrees, rmse_train_list, marker="o", label="Training RMSE")
plt.plot(degrees, rmse_test_list, marker="o", label="Test RMSE")
plt.yscale("log")
plt.xlabel("Polynomial degree")
plt.ylabel("RMSE (log scale)")
plt.title("Training vs. test error by polynomial degree")
plt.xticks(degrees)
plt.legend()
plt.show()

best_degree = degrees[int(np.argmin(rmse_test_list))]
print("Degree with the lowest test RMSE:", best_degree)

**Result:** the **test RMSE is lowest at degree 2**, which is exactly the degree of the true model **y = −x²**. 

- Training RMSE keeps falling as the degree grows, reaching ~0 at degree 11, where the polynomial passes through all 12 training points.
- Test RMSE behaves very differently: high at degree 1 (underfitting), **lowest at degree 2**, then it explodes, reaching millions at degree 11 (overfitting). That's why the y-axis needs a log scale.
- The gap between training and test error is the clearest sign of overfitting. A model that looks perfect on training data can be useless on new data.

This is the core idea behind using **validation / test data** to choose model complexity, and it applies directly to deep neural networks. Bigger networks can memorise training data, so we watch the validation loss and use techniques like regularisation, dropout and early stopping.

*Note: strictly speaking, this uses a single held-out test set rather than k-fold cross-validation. With k-fold, we'd repeat the split several times and average the errors, which gives a more stable choice.*